# Healthcare Demo — Orchestrator
Runs all five data-build notebooks in sequence. Each step is timed and logged. The run stops immediately if any notebook raises an exception.

In [2]:
from datetime import datetime
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# ============================================================
# Healthcare Demo - Orchestrator
#
# Runs all five data-build notebooks in sequence.
# Each step is timed and logged.
# Execution stops immediately if a notebook fails.
# ============================================================


# Configuration
# Timeout is applied individually to each referenced notebook.
TIMEOUT_SECONDS = 1800

# All notebooks are expected to exist in the same workspace.
NOTEBOOKS = [
    "01_create_provider_patient",
    "02_create_appointment",
    "03_create_diagnosis",
    "04_create_prescription",
    "05_validate_and_sample_queries",
]



StatementMeta(, 0b50c1e6-bf6b-474c-bbe7-b81a4f27f586, 4, Finished, Available, Finished, False)

In [3]:
# ============================================================
# Run notebooks
# ============================================================

run_log = []
orchestration_start = datetime.now()

print(
    f"Orchestration started at "
    f"{orchestration_start.strftime('%Y-%m-%d %H:%M:%S')}\n"
)

for step, nb_name in enumerate(NOTEBOOKS, start=1):

    step_start = datetime.now()
    status = "RUNNING"

    print(f"[{step}/{len(NOTEBOOKS)}] Starting: {nb_name} ...")

    try:
        exit_value = notebookutils.notebook.run(
            nb_name,
            TIMEOUT_SECONDS,
            {}
        )

        elapsed = int(
            (datetime.now() - step_start).total_seconds()
        )

        status = "PASS"

        print(f"  Completed in {elapsed}s")

        if exit_value not in [None, ""]:
            print(f"  Exit value: {exit_value}")

        print()

    except Exception as ex:

        elapsed = int(
            (datetime.now() - step_start).total_seconds()
        )

        status = "FAIL"

        run_log.append({
            "step": step,
            "notebook": nb_name,
            "status": status,
            "elapsed_seconds": elapsed,
            "error": str(ex)
        })

        print(
            f"  FAILED after {elapsed}s - stopping orchestration.\n"
        )
        print(f"  Error: {ex}\n")

        break

    run_log.append({
        "step": step,
        "notebook": nb_name,
        "status": status,
        "elapsed_seconds": elapsed,
        "error": ""
    })


# ============================================================
# Overall result
# ============================================================

total_elapsed = int(
    (datetime.now() - orchestration_start).total_seconds()
)

overall_status = (
    "PASS"
    if len(run_log) == len(NOTEBOOKS)
    and all(r["status"] == "PASS" for r in run_log)
    else "FAIL"
)

print(
    f"Orchestration {overall_status} - total {total_elapsed}s"
)


StatementMeta(, 0b50c1e6-bf6b-474c-bbe7-b81a4f27f586, 5, Finished, Available, Finished, False)

Orchestration started at 2026-10-23 14:22:23

[1/5] Starting: 01_create_provider_patient ...


  Completed in 49s

[2/5] Starting: 02_create_appointment ...


  Completed in 22s

[3/5] Starting: 03_create_diagnosis ...


  Completed in 18s

[4/5] Starting: 04_create_prescription ...


  Completed in 21s

[5/5] Starting: 05_validate_and_sample_queries ...


  Completed in 72s

Orchestration PASS - total 184s


In [4]:
# ============================================================
# Execution summary
# ============================================================

summary_schema = StructType([
    StructField("step", IntegerType(), True),
    StructField("notebook", StringType(), True),
    StructField("status", StringType(), True),
    StructField("elapsed_seconds", IntegerType(), True),
    StructField("error", StringType(), True)
])

summary_df = spark.createDataFrame(
    run_log,
    schema=summary_schema
)

summary_df = summary_df.select(
    "step",
    "notebook",
    "status",
    "elapsed_seconds",
    "error"
)

display(summary_df.orderBy("step"))


StatementMeta(, 0b50c1e6-bf6b-474c-bbe7-b81a4f27f586, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, bcd10ae5-2462-483d-bbdd-c09df5bc9aec)

In [5]:
# ============================================================
# Final status
#
# Raise an exception so a Fabric Data Pipeline calling this
# notebook receives a failure signal.
# ============================================================

if overall_status == "FAIL":

    failed = [
        r for r in run_log
        if r["status"] == "FAIL"
    ]

    raise RuntimeError(
        f"{len(failed)} notebook(s) failed: "
        f"{[f['notebook'] for f in failed]}"
    )

print("All notebooks completed successfully.")

StatementMeta(, 0b50c1e6-bf6b-474c-bbe7-b81a4f27f586, 7, Finished, Available, Finished, False)

All notebooks completed successfully.
